In [0]:
dbutils.widgets.text("catalog","ai")
dbutils.widgets.text("controlTableSchema","00_etlcontrol")
dbutils.widgets.text("env","dev")
dbutils.widgets.text("source","tutorial")


env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
controlTableSchema = dbutils.widgets.get("controlTableSchema")
source = dbutils.widgets.get("source")


In [0]:
from pyspark.sql.functions import pandas_udf, col,expr, sha2, current_user, current_timestamp, lit, date_format, when, to_json, struct
from typing import Iterator
import pandas as pd
from pyspark.sql.types import ArrayType, StringType, BinaryType, StructType, StructField, LongType, BooleanType
import io, os, re, sys
from pypdf import PdfReader
from pyspark.sql import SparkSession
import pymupdf
from datetime import datetime
import pytz 
import re

project_root = os.path.abspath("../../") 
if project_root not in sys.path:
    sys.path.append(project_root)

# Reload module to pick up any changes during development
import importlib
import lib.common.sparkConfig
import lib.bronze.autoloaderconfig
import lib.bronze.bronzevalidator

importlib.reload(lib.common.sparkConfig)
importlib.reload(lib.bronze.autoloaderconfig)
importlib.reload(lib.bronze.bronzevalidator)

from  lib.bronze.autoloaderconfig import AutoLoaderInitialize
from lib.bronze.bronzevalidator import Validator
from lib.common.sparkConfig import SourceInitialize




In [0]:
# from pyspark.sql.types import StructType, StructField, LongType, StringType

def getNotebookName():
        """Return the current notebook's file name (last segment of its workspace path)."""
        current_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return current_notebook.split("/")[-1] if current_notebook.split("/") else current_notebook

spark = SparkSession.builder.appName(getNotebookName()).getOrCreate()


class BronzeETL:
        """Bronze-layer ETL for landing-zone PDFs: Auto Loader ingestion, PDF validation,
        rejected-record capture, and audit logging, all driven by control tables.
        """
        
        def __init__(self, spark, src: SourceInitialize, ldr: AutoLoaderInitialize):
                """Initialize the BronzeETL processor.

                Args:
                    spark: active SparkSession for the notebook.
                    src: initialized SourceInitialize (env/notebook/source config from control tables).
                    ldr: initialized AutoLoaderInitialize (Auto Loader, file-meta and target properties).

                Loads Auto Loader config/file-meta/target-table properties, source properties,
                metadata select expressions, bronze write options, and the rejected-records table
                name from the control tables.
                """
                self.spark = spark
                self.src = src
                self.autoLoaderConfigProperties = ldr.getAutoloaderConfigProperties()
                self.autoLoaderFileMetaProperties = ldr.getAutoloaderFileMetaProperties()
                self.autoloaderTagetTableProperties = ldr.getAutoloaderTagetTableProperties()
                self.autoloaderTagetPartitionColumn = ldr.getAutoloaderTagetPartitionColumn()
                self.sourceProperties = src.getSourceProperties()
                self.metaColumns = self.getFileMetaData()
                self.sourceName = src.getSourceName()
                self.loadid = src.getLoadId()
                self.rejectedRecordsTable = src.getRejectedRecordsTable()
                self.inline_threshold_bytes = self.sourceProperties.get("INLINE_THRESHOLD_BYTES")
                self.bronzeTableProperties = src.getBronzeProperties()

        def getFileMetaData(self):
                """Build Auto Loader file-metadata select expressions.

                Returns:
                    list[str]: one '<value> as <name>' expression per file-metadata property
                    (e.g. file modification time), appended to ingested rows via selectExpr.
                """

                try:
                        return [f'{val} as {name}' for name,val in self.autoLoaderFileMetaProperties.items()]
                
                except Exception as e:
                        print(f"Error in getFileMetaData: {e}")
                        raise
        

        def hashDocument(self,batch_df):
                """Add a documentKey column: the SHA-256 hex digest of the file content,
                used as a stable document identity for downstream dedup.

                Returns:
                    DataFrame: input batch plus the documentKey column.
                """

                return (batch_df.select("*",sha2(col("content"), 256).alias("documentKey")))
               

        def logAudit(self, batch_id, cnt_total, cnt_valid, cnt_invalid):
                """Append one audit row for a micro-batch to <catalog>.<schema>.<layer>_auditlog.

                Records loadid, batchid, sourcename, total/valid/invalid counts, status
                (FAILURE when any invalid records exist, else SUCCESS), an error message,
                a UTC timestamp, and the creating user. On serverless,
                spark.table().schema drops delta.identity metadata, so identity columns are
                detected from SHOW CREATE TABLE DDL and excluded from the insert schema.
                """

                try: 

                # targetProps = self.autoloaderTagetTableProperties
                        if cnt_invalid > 0:
                                status = "FAILURE"
                                error_message = "CHECK BAD RECORDS"
                        else:
                                status = "SUCCESS"
                                error_message = None

                        AUDIT_TABLE = f'{self.autoloaderTagetTableProperties.get("targetCatelog")}.{self.autoloaderTagetTableProperties.get("targetSchema")}.{self.autoloaderTagetTableProperties.get("layer")}_auditlog'

                        table_schema = self.spark.table(AUDIT_TABLE).schema

                        # On serverless, spark.table().schema drops the delta.identity metadata,
                        # so identity columns are also detected from SHOW CREATE TABLE DDL
                        ddl = self.spark.sql(f"SHOW CREATE TABLE {AUDIT_TABLE}").collect()[0][0]
                        identity_columns = set(re.findall(
                                r'^\s*`?(\w+)`?\s+\w+.*GENERATED\s+(?:ALWAYS|BY DEFAULT)\s+AS\s+IDENTITY',
                                ddl, re.IGNORECASE | re.MULTILINE))

                        schema = StructType(
                                [f for f in table_schema.fields
                                if not any(k.startswith("delta.identity") for k in f.metadata)
                                and f.name not in identity_columns]
                                )

                        # current_user() gives the notebook user; the USER env var is unset in the
                        # foreachBatch worker (or holds the internal spark service identity)
                        created_by = self.spark.sql("SELECT current_user() AS user").collect()[0][0]

                        audit_records = [(
                                        self.loadid,
                                        str(batch_id),
                                        self.sourceName,
                                        str(cnt_total),
                                        str(cnt_valid),
                                        str(cnt_invalid),
                                        status,
                                        error_message,
                                        datetime.now(pytz.utc),
                                        created_by
                                        )]

                        

                        (self.spark.createDataFrame(audit_records, schema=schema)
                                .write.format("delta")
                                .mode("append")
                                .saveAsTable(AUDIT_TABLE)
                        )
                except Exception as e:
                        print(f"Error in logAudit: {e}")
                        raise

        def writeBronze(self, batch_df ):
                """Write valid records (isvalid == True) to the bronze document table
                <catalog>.<schema>.<layer>_<source>_document, using the configured
                format, mode, and bronze table properties.
                """
                try: 
                        (batch_df.filter(col("isvalid")==True)
                                .write
                                .format(f'{self.autoloaderTagetTableProperties.get("targetType")}')
                                .mode(f'{self.autoloaderTagetTableProperties.get("mode")}')
                                .options(**self.bronzeTableProperties)
                                .saveAsTable(f'{self.autoloaderTagetTableProperties.get("targetCatelog")}.{self.autoloaderTagetTableProperties.get("targetSchema")}.{self.autoloaderTagetTableProperties.get("layer")}_{ self.sourceName}_document')
                                        
                                )
                except Exception as e:
                        print(f"Error in writeBronze: {e}")
                        raise

        def writeBadRecordsBronze(self, batch_df ):
                """Write invalid records (isvalid == False), prefixed with a sourcename column,
                to the rejected-records table <catalog>.<schema>.<layer>_<source>_rejected_Records,
                using the configured format, mode, and bronze table properties.
                """

                try:
                        (batch_df.filter(col("isvalid")==False)
                                .select(lit(self.sourceName).alias("sourcename"),"*")
                                .write
                                .format(f'{self.autoloaderTagetTableProperties.get("targetType")}')
                                .mode(f'{self.autoloaderTagetTableProperties.get("mode")}')
                                .options(**self.bronzeTableProperties)
                                .saveAsTable(f'{self.autoloaderTagetTableProperties.get("targetCatelog")}.{self.autoloaderTagetTableProperties.get("targetSchema")}.{self.autoloaderTagetTableProperties.get("layer")}_{ self.sourceName}_rejected_Records')
                                        
                                )
                except Exception as e:
                        print(f"Error in writeBadRecordsBronze: {e}")
                        raise
        
        def getRecordCount(self, batch_df) -> tuple:
                """Compute record counts for a batch based on the isvalid flag.

                Returns:
                    tuple: (total, valid, invalid) row counts.
                """

                try:

                        cnt_total = batch_df.select("path").count()
                        cnt_valid = batch_df.filter(col("isvalid")==True).select("path").count()
                        cnt_invalid = batch_df.filter(col("isvalid")==False).select("path").count() 
                        
                        return (cnt_total, cnt_valid, cnt_invalid)
                
                except Exception as e:
                        print(f"Error in getRecordCount: {e}")
                        raise

        def processBatch(self,batch_df, batch_id):
                """Process one streaming micro-batch (foreachBatch callback).

                Skips empty batches. Otherwise stamps each row with batchid and loadid,
                adds the SHA-256 documentKey, then validates every file via mapInPandas
                (mime-type detection plus PDF checks: encryption, %%EOF marker, page count,
                file size), producing isvalid, validation_checks, and mime_type columns.
                Writes valid rows to the bronze document table, invalid rows to the
                rejected-records table, and appends an audit-log row.
                """

                try: 
                        if batch_df.isEmpty():
                                return
                        
                        targetProps = self.autoloaderTagetTableProperties
                                
                        batch_df = batch_df.select("*", lit(batch_id).alias("batchid"), lit(self.loadid).alias("loadid"))
                        batch_df = self.hashDocument(batch_df)
                        batch_schema = batch_df.schema
                        schema = (batch_schema.add( StructField("isvalid", BooleanType(), True)) 
                                                .add( StructField("validation_checks", ArrayType(StructType([StructField("type", StringType()),StructField("isvalid", BooleanType()), StructField("error", StringType())])), True))
                                                .add(StructField("mime_type", StringType(), True))
                                                )
                        # batch_df = batch_df.withColumn("isvalid", lit(True)
                        validator = Validator()
                        batch_df = batch_df.mapInPandas(
                                        validator.validate_batch,
                                        schema=schema
                                        )
                        
                        batch_df = batch_df.drop("content")
                        
                        self.writeBronze(batch_df)  
                        self.writeBadRecordsBronze( batch_df)           
                        self.logAudit(batch_id, self.getRecordCount(batch_df)[0], self.getRecordCount(batch_df)[1], self.getRecordCount(batch_df)[2])
                
                except Exception as e:
                        print(f"Error in processBatch: {e}")
                        raise


                
        def bronzeLoad(self):
                """Start the bronze Auto Loader streaming ingestion.

                Reads new files from the configured landing path using cloudFiles with
                control-table-driven options, enriches rows with file-metadata columns, and
                processes each micro-batch through processBatch via foreachBatch with a
                durable checkpoint location. Uses an availableNow trigger (drain then stop)
                and blocks until the query terminates.
                """

                try:
                        query = (self.spark.readStream.format("cloudFiles") 
                        .options(**self.autoLoaderConfigProperties)
                        .load(self.sourceProperties.get("file_path"))
                        .selectExpr("*", *self.metaColumns)
                        .writeStream
                        .foreachBatch(self.processBatch)
                        .option("checkpointLocation", f"{self.sourceProperties.get('checkpint')}/{self.sourceName}")
                        .trigger(availableNow=True)
                        .start()
                        )

                        query.awaitTermination()
                except Exception as e:
                        print(f"Error in bronzeLoad: {e}")
                        raise
                
                
if __name__ == "__main__":

        try:

                print(source)
                src = SourceInitialize(spark, getNotebookName(), env, catalog, controlTableSchema, source)
                src.checkSetup()

                ldr = AutoLoaderInitialize(spark, catalog, controlTableSchema,src.getAutoloaderPropertiesTable(), src.getSourceId())
                ldr.checkSetup()

                bz_etl = BronzeETL(spark, src, ldr)        
                bz_etl.bronzeLoad()
        except Exception as e:
                print(f"Error in main: {e}")
                raise

In [0]:
df = (spark.read.table("ai.`02_bronze`.bronze_tutorial_document"))

display(df.drop("content"))

df = spark.read.table("ai.`02_bronze`.bronze_auditlog")
display(df)

df = spark.read.table("ai.`02_bronze`.bronze_tutorial_rejected_records")
display(df)

